In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
sns.set_theme(style='whitegrid')

PASTA_DADOS = Path('dados_tratados')
META_ANUAL = 88000
META_COLABORATIVA = 70000

#   confirmar de onde vem o valor real de engajamento com o MEJ?
#   idem para "Tempo de Permanência no MEJ" e "Políticas de Diversidade e Inclusão"
ENG_MEJ_PCT_ASSUMIDO = 0.70          # 0.70 = 70% de engajamento (valor ilustrativo, igual ao exemplo do PDF)
CLUSTER_ANTERIOR_CONHECIDO = None    # cluster oficial do ano anterior — preencher se a Direx informar
SELO_EJ_EM_CONFORMIDADE = True       # confirmar com o gestor antes da entrega final

In [2]:
def carregar_bases(pasta=PASTA_DADOS):
    '''Carrega as saídas dos notebooks anteriores. Nenhuma delas precisa ser reexecutada
    para rodar este notebook — todas já existem em `dados_tratados/`.'''
    base_mensal = pd.read_csv(pasta / 'base_cluster_mensal.csv')
    projetos = pd.read_csv(pasta / 'projetos_tratados.csv', parse_dates=['data_inicio', 'data_fim', 'data_registro'])
    previsao = pd.read_csv(pasta / 'previsao_faturamento.csv')
    with open(pasta / 'trajetoria_metas.json', encoding='utf-8') as f:
        trajetoria_metas = json.load(f)
    return base_mensal, projetos, previsao, trajetoria_metas

base_mensal, projetos, previsao, trajetoria_metas = carregar_bases()
projetos[['nome_projeto', 'ano_inicio', 'status', 'satisfacao', 'valor_total_projeto', 'projeto_de_impacto']].tail()

,nome_projeto,ano_inicio,status,satisfacao,valor_total_projeto,projeto_de_impacto
17,Projeto Fênix,2026,Em andamento,NaN,10000,True
18,Projeto Quasar,2026,Iniciado,NaN,2000,False
19,Projeto ASURA,2026,Iniciado,NaN,4000,False
20,Projeto Arthur,2026,Iniciado,NaN,5000,False
21,Projeto Eclipse,2026,Em andamento,NaN,30000,False


In [ ]:
REGUA_CLUSTER = [
    (0,             12_000_000,   1),
    (12_000_000.01, 24_000_000,   2),
    (24_000_000.01, 61_000_000,   3),
    (61_000_000.01, 130_000_000,  4),
    (130_000_000.01, float('inf'), 5),
]

def classificar_cluster(indice):
    '''Mapeia um índice numérico para o cluster (1 a 5) conforme a régua oficial.'''
    for minimo, maximo, cluster in REGUA_CLUSTER:
        if minimo <= indice <= maximo:
            return cluster
    return None

pd.DataFrame(REGUA_CLUSTER, columns=['minimo', 'maximo', 'cluster'])

In [ ]:
def calcular_indice_cluster(faturamento, csat, eng_mej_pct, pct_faturamento_colaborativo):
    '''
    Índice = Faturamento x CSAT x (1 + %Engajamento MEJ) x (1 + %Faturamento Colaborativo) x 100
    (fórmula tirada literalmente da Calculadora de Cluster do PDF do projeto).
    '''
    pct_faturamento_colaborativo = 0 if pd.isna(pct_faturamento_colaborativo) else pct_faturamento_colaborativo
    eng_mej_pct = 0 if pd.isna(eng_mej_pct) else eng_mej_pct
    csat = 0 if pd.isna(csat) else csat
    return faturamento * csat * (1 + eng_mej_pct) * (1 + pct_faturamento_colaborativo) * 100

# Validação com o exemplo do próprio PDF: Faturamento R$ 88.000, CSAT 5, Eng.MEJ 1.70, Fat.Colab 1.80
indice_exemplo = calcular_indice_cluster(88000, 5, 0.70, 0.80)
print(f'Índice de exemplo: {indice_exemplo:,.0f}  →  cluster {classificar_cluster(indice_exemplo)}')
print('Esperado no PDF: 134.300.000,00 (a pequena diferença é arredondamento do próprio exemplo)')

In [ ]:
projetos['ano'] = projetos['ano_inicio']

resumo_anual = projetos.groupby('ano').agg(
    faturamento_total=('valor_total_projeto', 'sum'),
    faturamento_colaborativo=('valor_faturamento_colaborativo', 'sum'),
    qtd_projetos_impacto=('projeto_de_impacto', 'sum'),
    csat_medio=('satisfacao', 'mean'),
).reset_index()
resumo_anual['pct_faturamento_colaborativo'] = (
    resumo_anual['faturamento_colaborativo'] / resumo_anual['faturamento_total']
)

# Anos/meses sem avaliação de CSAT concluída usam a média histórica geral como aproximação
# (ex.: 2026 — todos os projetos do ano ainda estão "Em andamento"/"Iniciado", sem CSAT coletado).
CSAT_HISTORICO_GERAL = projetos['satisfacao'].mean()
print(f'CSAT histórico geral (fallback p/ anos sem avaliação concluída): {CSAT_HISTORICO_GERAL:.2f}')

def avaliar_ano(row, eng_mej_pct=ENG_MEJ_PCT_ASSUMIDO, cluster_anterior=None,
                 selo_ej_ok=True, csat_fallback=CSAT_HISTORICO_GERAL):
    csat_usado = row['csat_medio'] if pd.notna(row['csat_medio']) else csat_fallback
    indice = calcular_indice_cluster(row['faturamento_total'], csat_usado, eng_mej_pct,
                                      row['pct_faturamento_colaborativo'])
    cluster_bruto = classificar_cluster(indice)
    cluster_final, motivos = aplicar_regras_cluster(
        cluster_bruto, row['faturamento_total'], row['qtd_projetos_impacto'], cluster_anterior, selo_ej_ok
    )
    return pd.Series({
        'csat_usado': csat_usado,
        'indice_cluster': indice,
        'cluster_bruto': cluster_bruto,
        'cluster_final': cluster_final,
        'motivos_ajuste': ', '.join(motivos) if motivos else None,
    })

avaliacao_hist = resumo_anual.join(resumo_anual.apply(avaliar_ano, axis=1))
avaliacao_hist